# Parte 1: Recolección y exploración de noticias peruanas

## Configuración inicial

- **Estudiante:** Diego Enrique Sime Rendón.
- **Medio elegido:** La República — opción A.
- **Fecha de descarga:** pendiente; registrar al ejecutar la recolección.
- **Sitio base:** https://larepublica.pe/
- **URLs específicas de extracción:** pendientes de verificar.
- **Meta:** 400 noticias válidas, aproximadamente 100 por categoría.
- **Categorías:** `politica`, `economia`, `deportes`, `ciencia_tecnologia`.

## Pregunta de esta etapa

¿Cómo construir una base de noticias reales, sin duplicados y con categorías
normalizadas, que pueda utilizarse posteriormente para entrenar y evaluar
un clasificador?

## Procesos previstos

1. Verificar las fuentes y las condiciones de acceso.
2. Extraer noticias y conservar un respaldo original.
3. Explorar los datos e identificar problemas.
4. Limpiar y normalizar las categorías.
5. Elaborar tres gráficos exploratorios.
6. Validar y exportar la base final.

In [1]:
# Identificar el Python que ejecuta este notebook.
import sys

print("Versión:", sys.version)
print("Ubicación:", sys.executable)


Versión: 3.12.15 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:14:50) [MSC v.1942 64 bit (AMD64)]
Ubicación: c:\Users\Diego Sime\.conda\envs\pucp\python.exe


In [2]:
# PROCESO 0: Comprobar las herramientas del entorno.
# Insumo: entorno Python seleccionado.
# Resultado: listado de paquetes disponibles y faltantes.

# find_spec permite comprobar si un paquete está disponible sin cargarlo.
from importlib.util import find_spec

paquetes = {
    "requests": "requests",
    "beautifulsoup4": "bs4",
    "pandas": "pandas",
    "plotly": "plotly",
}

for nombre, modulo in paquetes.items():
    estado = "Disponible" if find_spec(modulo) is not None else "Falta instalar"
    print(f"{nombre}: {estado}")

requests: Disponible
beautifulsoup4: Disponible
pandas: Disponible
plotly: Disponible


In [3]:
# Instalar Plotly en el entorno que ejecuta este notebook: pucp.
# %pip install plotly

In [4]:
# Cargar Plotly Express para crear gráficos interactivos.
import plotly.express as px

print("Plotly Express cargado correctamente")

Plotly Express cargado correctamente


## Proceso 1: Verificar las condiciones de acceso

**Insumo:** archivo robots.txt de La República.

**Método:** consultar las instrucciones publicadas para programas automatizados.

**Resultado esperado:** identificar las rutas permitidas o restringidas antes
de decidir cómo recolectar las noticias.

In [5]:
# PROCESO 1: Consultar las condiciones de acceso.
# Insumo: URL del archivo robots.txt.
# Resultado: estado de la solicitud y contenido del archivo.

# requests: permite solicitar información a un sitio web.
import requests

url_robots = "https://larepublica.pe/robots.txt"

# User-Agent: identifica nuestro programa ante el sitio.
cabeceras = {
    "User-Agent": "ProyectoAcademicoNoticias/1.0 (curso Python; uso educativo)"
}

# get(): solicita el archivo.
# timeout=30: limita la espera de respuesta a 30 segundos.
respuesta_robots = requests.get(
    url_robots,
    headers=cabeceras,
    timeout=30
)

print("Estado HTTP:", respuesta_robots.status_code)

# Si la solicitud falló, detiene esta celda con un mensaje de error.
respuesta_robots.raise_for_status()

# .text: obtiene el contenido de la respuesta como texto.
print(respuesta_robots.text)

Estado HTTP: 200
User-agent: *
User-agent: Googlebot
User-agent: Googlebot-News
User-agent: Googlebot-Image
User-agent: Googlebot-Video
User-agent: Googlebot-Mobile
User-agent: PetalBot
User-agent: Bingbot
User-agent: OAI-SearchBot
User-agent: ChatGPT-User
User-agent: GPTBot
User-agent: Amazonbot
User-agent: anthropic-ai
User-agent: AwarioRssBot
User-agent: AwarioSmartBot
User-agent: Bytespider
User-agent: CCBot
User-agent: ClaudeBot
User-agent: Claude-Web
User-agent: cohere-ai
User-agent: DataForSeoBot
User-agent: Diffbot
User-agent: ImagesiftBot
User-agent: FacebookBot
User-agent: Google-Extended
User-agent: magpie-crawler
User-agent: NewsNow
User-agent: news-please
User-agent: omgili
User-agent: omgilibot
User-agent: PerplexityBot
User-agent: PiplBot
User-agent: Scrapy
User-agent: TurnitinBot
User-agent: YouBot

#Info private
Disallow: /buscador*
Disallow: /data
Disallow: /data/*
Disallow: /contactenos
Disallow: /*/pag/
Disallow: /*/pag/*
Disallow: /*/*/page/*
Disallow: /archive/*
D

### Decisión sobre la fuente

El archivo robots.txt anuncia y permite rutas RSS. Se comenzará probando
los RSS de política, economía, deportes y ciencia.

Se verificará su contenido y cobertura antes de decidir si permiten alcanzar
la cantidad requerida. Se respetarán pausas de 1 a 2 segundos entre solicitudes.

In [6]:
# PROCESO 1.2: Inspeccionar una fuente RSS.
# Insumo: URL del RSS de política.
# Resultado: estado, tipo de contenido y una muestra del texto recibido.

# time: permite introducir una pausa entre solicitudes.
import time

url_rss_politica = "https://larepublica.pe/rss/politica.xml"

# Esperar dos segundos antes de realizar otra solicitud al diario.
time.sleep(2)

respuesta_rss = requests.get(
    url_rss_politica,
    headers=cabeceras,
    timeout=30
)

print("Estado HTTP:", respuesta_rss.status_code)
print("Tipo de contenido:", respuesta_rss.headers.get("Content-Type"))

respuesta_rss.raise_for_status()

# Mostrar solo los primeros 2.000 caracteres para inspeccionar su estructura.
print(respuesta_rss.text[:2000])

Estado HTTP: 200
Tipo de contenido: text/xml; charset=UTF-8
<?xml version="1.0" encoding="UTF-8"?>
    
            <rss xmlns:image="http://www.google.com/schemas/sitemap-image/1.1" xmlns:media="http://search.yahoo.com/mrss/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:content="http://purl.org/rss/1.0/modules/content/" version="2.0">
                <channel>
                <title>La República: Últimas noticias de última hora del Perú y el mundo</title>
                <link>https://larepublica.pe</link>
                <description>Noticias del Perú y del mundo en larepublica.pe - Últimas noticias de política, espectáculos, deportes, economía, tendencias, tecnología, salud, sociedad, mundo, cine y más.</description>
                <lastBuildDate>Sun, 18 Oct 2026 18:20:52 GMT</lastBuildDate>
                <language>es</language>
                    
                        <item>
                            <title>
                                <![CDATA[ Jorge Nieto plante

In [7]:
# PROCESO 1.3: Conservar e inspeccionar el RSS recibido.
# Insumo: respuesta_rss, ya descargada.
# Resultado: respaldo XML, cantidad de noticias y campos del primer registro.

# pathlib: permite trabajar con carpetas y archivos.
from pathlib import Path

# ElementTree: permite leer la estructura XML; viene incluido con Python.
import xml.etree.ElementTree as ET

# Identificar la carpeta de la tarea según dónde se ejecute el notebook.
carpeta_actual = Path.cwd()

if carpeta_actual.name == "tarea_1":
    carpeta_tarea = carpeta_actual
else:
    carpeta_tarea = carpeta_actual / "tarea_1"

# Evitar guardar en una ubicación equivocada.
if not carpeta_tarea.is_dir():
    raise FileNotFoundError(
        f"No se encontró tarea_1. Carpeta actual: {carpeta_actual}"
    )

carpeta_raw = carpeta_tarea / "datos" / "raw"
carpeta_raw.mkdir(parents=True, exist_ok=True)

# Guardar los bytes originales, antes de transformar la información.
archivo_rss = carpeta_raw / "politica_inicial.xml"
archivo_rss.write_bytes(respuesta_rss.content)

print("Respaldo guardado en:", archivo_rss)

# Convertir el XML recibido en una estructura que Python pueda recorrer.
raiz_rss = ET.fromstring(respuesta_rss.content)

# Buscar todos los bloques item dentro del canal.
noticias_rss = raiz_rss.findall("./channel/item")

print("Noticias encontradas:", len(noticias_rss))

# Inspeccionar los campos de la primera noticia.
if noticias_rss:
    print("\nCampos de la primera noticia:")

    for campo in noticias_rss[0]:
        contenido = "".join(campo.itertext()).strip()
        print(f"{campo.tag}: {contenido[:300]}")

Respaldo guardado en: d:\Mis Documentos\GitHub\Python-Intermedio-QLAB\tarea_1\datos\raw\politica_inicial.xml
Noticias encontradas: 19

Campos de la primera noticia:
title: Jorge Nieto plantea aprobar las 20 facultades respaldadas por Diputados y tramitar las adicionales como proyectos de ley
link: https://larepublica.pe/politica/2026/10/18/jorge-nieto-plantea-aprobar-las-20-facultades-respaldadas-por-diputados-y-tramitar-las-adicionales-como-proyectos-de-ley-hnews-1130742
guid: https://larepublica.pe/politica/2026/10/18/jorge-nieto-plantea-aprobar-las-20-facultades-respaldadas-por-diputados-y-tramitar-las-adicionales-como-proyectos-de-ley-hnews-1130742
description: El líder del Partido del Buen Gobierno señaló que hizo llegar esta propuesta a representantes del Gobierno de Keiko Fujimori. Además, pidió esclarecer la dinámica entre cámaras del Parlamento para evitar enfrentamientos.
{http://www.google.com/schemas/sitemap-image/1.1}image: https://larepublica.cronosmedia.glr.pe/original/2

In [8]:
# PROCESO 2: Convertir las noticias del RSS en una tabla.
# Insumo: noticias_rss, los 20 elementos XML ya descargados.
# Resultado: tabla noticias_politica con las siete columnas requeridas.

# pandas: organiza los registros en filas y columnas.
import pandas as pd

# BeautifulSoup: permite quitar posibles etiquetas HTML de la descripción.
from bs4 import BeautifulSoup

registros = []

for noticia in noticias_rss:
    # findtext(): obtiene el texto del elemento indicado.
    titular = noticia.findtext("title", default="").strip()
    descripcion = noticia.findtext("description", default="").strip()

    # get_text(): conserva el texto de la descripción sin etiquetas HTML.
    bajada = BeautifulSoup(
        descripcion, "html.parser"
    ).get_text(" ", strip=True)

    registros.append({
        "titular": titular,
        "bajada": bajada,
        "medio": "La Republica",
        "seccion_original": noticia.findtext(
            "category", default=""
        ).strip(),
        "url": noticia.findtext("link", default="").strip(),
        # Conservamos por ahora el formato original del RSS.
        "fecha": noticia.findtext("pubDate", default="").strip(),
        # Asignamos la categoría solo si encontramos la sección esperada.
        "etiqueta": (
            "politica"
            if noticia.findtext("category", default="").strip() == "Política"
            else None
        )
    })

# Cada diccionario se convierte en una fila.
noticias_politica = pd.DataFrame(registros)

print("Dimensiones (filas, columnas):", noticias_politica.shape)

print("\nSecciones encontradas:")
print(noticias_politica["seccion_original"].value_counts(dropna=False))

display(noticias_politica.head())

Dimensiones (filas, columnas): (19, 7)

Secciones encontradas:
seccion_original
Política    19
Name: count, dtype: int64


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Jorge Nieto plantea aprobar las 20 facultades ...,El líder del Partido del Buen Gobierno señaló ...,La Republica,Política,https://larepublica.pe/politica/2026/10/18/jor...,"Sun, 18 Oct 2026 18:20:52 GMT",politica
1,Comisión de Constitución del Senado continuará...,El debate se reanudará desde las 11 de la maña...,La Republica,Política,https://larepublica.pe/politica/2026/10/18/com...,"Sun, 18 Oct 2026 17:31:06 GMT",politica
2,Carlos Infante: “La matanza de Uchuraccay es u...,"El periodista Carlos Infante, entre la memoria...",La Republica,Política,https://larepublica.pe/politica/2026/08/15/car...,"Sun, 18 Oct 2026 15:33:36 GMT",politica
3,Fernando Cerimedo tenía operadores en las elec...,Conexiones gauchas. El consultor argentino pre...,La Republica,Política,https://larepublica.pe/politica/2026/10/18/fer...,"Sun, 18 Oct 2026 11:50:00 GMT",politica
4,Número de regiones que irán a reelección aumen...,Especialistas coinciden en que uno de los fact...,La Republica,Política,https://larepublica.pe/politica/2026/10/18/num...,"Sun, 18 Oct 2026 10:00:00 GMT",politica


## Proceso 3: Examinar una fuente complementaria

El RSS de política proporcionó 20 noticias, cantidad insuficiente para
la meta aproximada de 100 por categoría.

Se inspeccionará el sitemap de política anunciado en robots.txt para
determinar si permite localizar noticias adicionales.

In [9]:
# PROCESO 3: Inspeccionar el sitemap de política.
# Insumo: URL anunciada en robots.txt.
# Resultado: respaldo XML, estructura y primeros enlaces.

url_sitemap = "https://larepublica.pe/sitemap/politica.xml"

# Respetar una pausa entre solicitudes.
time.sleep(2)

respuesta_sitemap = requests.get(
    url_sitemap,
    headers=cabeceras,
    timeout=30
)

print("Estado HTTP:", respuesta_sitemap.status_code)
respuesta_sitemap.raise_for_status()

# Conservar el contenido original antes de procesarlo.
archivo_sitemap = carpeta_raw / "sitemap_politica_inicial.xml"
archivo_sitemap.write_bytes(respuesta_sitemap.content)

# Leer la estructura XML.
raiz_sitemap = ET.fromstring(respuesta_sitemap.content)

print("Elemento raíz:", raiz_sitemap.tag)

# {*} permite encontrar elementos aunque utilicen un espacio de nombres.
# Buscar entradas directas de noticias y posibles sitemaps secundarios.
entradas_noticias = raiz_sitemap.findall("{*}url")
entradas_sitemaps = raiz_sitemap.findall("{*}sitemap")

print("Entradas de noticias:", len(entradas_noticias))
print("Sitemaps secundarios:", len(entradas_sitemaps))

# Mostrar cinco enlaces para comprobar qué contiene.
entradas = entradas_noticias + entradas_sitemaps

print("\nPrimeros cinco enlaces:")
for entrada in entradas[:5]:
    print(entrada.findtext("{*}loc", default="Sin enlace"))

# Inspeccionar una entrada completa.
if entradas:
    print("\nEstructura de la primera entrada:")
    print(ET.tostring(entradas[0], encoding="unicode"))

Estado HTTP: 200
Elemento raíz: {http://www.sitemaps.org/schemas/sitemap/0.9}urlset
Entradas de noticias: 19
Sitemaps secundarios: 0

Primeros cinco enlaces:
https://larepublica.pe/politica/2026/10/18/jorge-nieto-plantea-aprobar-las-20-facultades-respaldadas-por-diputados-y-tramitar-las-adicionales-como-proyectos-de-ley-hnews-1130742
https://larepublica.pe/politica/2026/10/18/comision-de-constitucion-del-senado-continuara-debate-de-pedido-de-facultades-este-lunes-19-hnews-577638
https://larepublica.pe/politica/2026/08/15/carlos-infante-la-matanza-de-uchuraccay-es-un-golpe-terrible-a-la-memoria-de-los-periodistas-hnews-1405080
https://larepublica.pe/politica/2026/10/18/fernando-cerimedo-tenia-operadores-en-las-elecciones-del-peru-keiko-fujimori-federico-falco-fuerza-popular-hnews-879410
https://larepublica.pe/politica/2026/10/18/numero-de-regiones-que-iran-a-reeleccion-aumento-por-dispersion-de-voto-hnews-428587

Estructura de la primera entrada:
<ns0:url xmlns:ns0="http://www.sitemaps.

In [10]:
# PROCESO 3.1: Comparar cobertura y examinar el segundo sitemap.
# Insumos: noticias del RSS y entradas del sitemap ya descargado.
# Resultado: enlaces adicionales y estructura del sitemap web.

# Convertir los enlaces en conjuntos permite comparar URLs únicas.
urls_rss = set(noticias_politica["url"])

urls_sitemap = {
    entrada.findtext("{*}loc", default="").strip()
    for entrada in entradas_noticias
}
urls_sitemap.discard("")

print("URLs compartidas con el RSS:", len(urls_sitemap & urls_rss))
print("URLs adicionales del primer sitemap:", len(urls_sitemap - urls_rss))

# Probar la otra dirección anunciada en robots.txt.
url_sitemap_web = "https://larepublica.pe/sitemap/web/politica.xml"

time.sleep(2)

respuesta_sitemap_web = requests.get(
    url_sitemap_web,
    headers=cabeceras,
    timeout=30
)

print("\nEstado HTTP del sitemap web:", respuesta_sitemap_web.status_code)
respuesta_sitemap_web.raise_for_status()

# Guardar el respaldo original.
(carpeta_raw / "sitemap_web_politica_inicial.xml").write_bytes(
    respuesta_sitemap_web.content
)

raiz_web = ET.fromstring(respuesta_sitemap_web.content)

entradas_web = raiz_web.findall("{*}url")
sitemaps_secundarios_web = raiz_web.findall("{*}sitemap")

print("Elemento raíz:", raiz_web.tag)
print("Entradas de noticias:", len(entradas_web))
print("Sitemaps secundarios:", len(sitemaps_secundarios_web))

print("\nPrimeros cinco enlaces:")
for entrada in (entradas_web + sitemaps_secundarios_web)[:5]:
    print(entrada.findtext("{*}loc", default="Sin enlace"))

URLs compartidas con el RSS: 19
URLs adicionales del primer sitemap: 0

Estado HTTP del sitemap web: 200
Elemento raíz: {http://www.sitemaps.org/schemas/sitemap/0.9}urlset
Entradas de noticias: 19
Sitemaps secundarios: 0

Primeros cinco enlaces:
https://larepublica.pe/politica/2026/10/18/jorge-nieto-plantea-aprobar-las-20-facultades-respaldadas-por-diputados-y-tramitar-las-adicionales-como-proyectos-de-ley-hnews-1130742
https://larepublica.pe/politica/2026/10/18/comision-de-constitucion-del-senado-continuara-debate-de-pedido-de-facultades-este-lunes-19-hnews-577638
https://larepublica.pe/politica/2026/08/15/carlos-infante-la-matanza-de-uchuraccay-es-un-golpe-terrible-a-la-memoria-de-los-periodistas-hnews-1405080
https://larepublica.pe/politica/2026/10/18/fernando-cerimedo-tenia-operadores-en-las-elecciones-del-peru-keiko-fujimori-federico-falco-fuerza-popular-hnews-879410
https://larepublica.pe/politica/2026/10/18/numero-de-regiones-que-iran-a-reeleccion-aumento-por-dispersion-de-voto-

In [11]:
# PROCESO 3.3: Probar la consulta utilizada por "Ver más noticias".
# Insumo: ruta y parámetros identificados en el código de la página.
# Resultado: comprobar si recibimos datos utilizables con requests.

url_mas_noticias = "https://larepublica.pe/api/search/articles"

parametros = {
    "category_slug": "politica",
    "limit": 12,
    "page": 2,
    "order_by": "update_date",
    "view": "section"
}

time.sleep(2)

respuesta_mas = requests.get(
    url_mas_noticias,
    params=parametros,
    headers=cabeceras,
    timeout=30
)

print("Estado HTTP:", respuesta_mas.status_code)
print("Tipo de contenido:", respuesta_mas.headers.get("Content-Type"))

if respuesta_mas.status_code == 200:
    # Guardar la respuesta original antes de procesarla.
    (carpeta_raw / "politica_pagina_2_respuesta.txt").write_bytes(
        respuesta_mas.content
    )

    # Intentar interpretar la respuesta como JSON.
    try:
        datos_mas = respuesta_mas.json()

        print("Tipo de objeto recibido:", type(datos_mas).__name__)

        if isinstance(datos_mas, dict):
            print("Claves principales:", list(datos_mas.keys()))

        print("\nMuestra de la respuesta:")
        print(respuesta_mas.text[:2000])

    except ValueError:
        print("La respuesta no es JSON.")
        print(respuesta_mas.text[:500])

else:
    print("No se recibió una respuesta satisfactoria.")
    print("No repetiremos automáticamente esta solicitud.")

Estado HTTP: 403
Tipo de contenido: text/html
No se recibió una respuesta satisfactoria.
No repetiremos automáticamente esta solicitud.


In [12]:
# Instalar Selenium en el entorno activo del notebook.
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [13]:
# urljoin: convierte enlaces relativos en URLs completas.
# urlparse: separa una URL para revisar su dominio y su ruta.
from urllib.parse import urljoin, urlparse

In [14]:
# PROCESO 3.4: Comprobar la carga de Política en Chrome.
# Insumo: dirección de la sección Política.
# Resultado: título, dirección final y muestra del contenido visible.

# webdriver: permite controlar un navegador desde Python.
from selenium import webdriver

# By: permite localizar elementos dentro de la página.
from selenium.webdriver.common.by import By

# WebDriverWait: espera hasta que se cumpla una condición.
from selenium.webdriver.support.ui import WebDriverWait

# Abrir una ventana visible de Chrome.
navegador = webdriver.Chrome()

try:
    navegador.get("https://larepublica.pe/politica")

    # Esperar hasta que el cuerpo de la página tenga texto.
    WebDriverWait(navegador, 30).until(
        lambda web: web.find_element(By.TAG_NAME, "body").text.strip()
    )

    print("Título:", navegador.title)
    print("Dirección final:", navegador.current_url)

    texto_visible = navegador.find_element(By.TAG_NAME, "body").text
    print("\nPrimeros 1.000 caracteres:")
    print(texto_visible[:1000])

finally:
    # Cerrar Chrome incluso si ocurre un error.
    navegador.quit()

Título: Últimas Noticias de Política del Perú en La República
Dirección final: https://larepublica.pe/politica

Primeros 1.000 caracteres:
EN VIVO
PROCESIÓN DEL SEÑOR DE LOS MILAGROS EN VIVO: RECORRIDO DEL DOMINGO 18 DE OCTUBRE
Política
Keiko Fujimori
Comisión de Constitución
Rosa Bartra
Fernando Cerimedo tenía operadores en las elecciones del Perú
Comisión de Constitución del Senado continuará debate de pedido de facultades
Del TikTok al autoritarismo: Keiko Fujimori cambia de estrategia discursiva
León Trahtemberg: “No hay forma de escribir un libro si no hay ideología”
Fujimori ante el riesgo de un meganiño: “Más que eso no podemos hacer”
Más noticias
JORGE NIETO
Jorge Nieto plantea aprobar las 20 facultades respaldadas por Diputados y tramitar las adicionales como proyectos de ley
El líder del Partido del Buen Gobierno señaló que hizo llegar esta propuesta a representantes del Gobierno de Keiko Fujimori. Además, pidió esclarecer la dinámica entre cámaras del Parlamento para evitar 

In [15]:
# PROCESO 3.5: Comprobar el efecto de "Ver más noticias".
# Insumo: página de Política.
# Resultado: URLs únicas antes y después de un clic.

# re: reconoce las rutas de artículos de política.
import re

# TimeoutException: permite informar si el botón o las noticias no aparecen.
from selenium.common.exceptions import TimeoutException

# EC: permite esperar hasta que un botón esté visible y habilitado.
from selenium.webdriver.support import expected_conditions as EC


def obtener_urls_politica(web):
    """Devuelve un conjunto de URLs únicas de artículos de política."""
    # Leer el HTML actual, incluido el contenido añadido por JavaScript.
    contenido = BeautifulSoup(web.page_source, "html.parser")
    urls = set()

    for enlace in contenido.find_all("a", href=True):
        url_completa = urljoin(web.current_url, enlace["href"])
        partes = urlparse(url_completa)

        if (
            partes.hostname == "larepublica.pe"
            and re.match(r"^/politica/\d{4}/\d{2}/\d{2}/", partes.path)
        ):
            urls.add(partes._replace(query="", fragment="").geturl())

    return urls


navegador = webdriver.Chrome()

try:
    navegador.get("https://larepublica.pe/politica")

    # Esperar hasta encontrar enlaces a artículos.
    WebDriverWait(navegador, 30).until(obtener_urls_politica)

    # Localizar el botón por su texto, sin depender de clases de diseño.
    selector_boton = (
        "//*[self::button or self::a or @role='button']"
        "[contains(translate(normalize-space(.), "
        "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), "
        "'ver más noticias')]"
    )

    boton = WebDriverWait(navegador, 20).until(
        EC.element_to_be_clickable((By.XPATH, selector_boton))
    )

    urls_antes = obtener_urls_politica(navegador)
    print("URLs únicas antes del clic:", len(urls_antes))

    # Conservar el HTML antes de la interacción.
    (carpeta_raw / "politica_antes_del_clic.html").write_text(
        navegador.page_source, encoding="utf-8"
    )

    # Desplazar la página hasta situar el botón en el centro de la ventana.
    navegador.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        boton
    )

    # Dar tiempo a que termine el desplazamiento.
    time.sleep(2)

    # Intentar el clic normal.
    boton.click()

    # Esperar hasta que aparezca al menos una URL que antes no estaba.
    WebDriverWait(navegador, 30).until(
        lambda web: obtener_urls_politica(web) - urls_antes
    )

    urls_despues = obtener_urls_politica(navegador)
    urls_nuevas = urls_despues - urls_antes

    (carpeta_raw / "politica_despues_del_clic.html").write_text(
        navegador.page_source, encoding="utf-8"
    )

    print("URLs únicas después del clic:", len(urls_despues))
    print("URLs nuevas:", len(urls_nuevas))

    print("\nMuestra de URLs nuevas:")
    for url in sorted(urls_nuevas)[:5]:
        print(url)

except TimeoutException:
    print("No se encontró el botón a tiempo o no aparecieron URLs nuevas.")
    print("Título de la página:", navegador.title)

finally:
    navegador.quit()

URLs únicas antes del clic: 14
URLs únicas después del clic: 25
URLs nuevas: 11

Muestra de URLs nuevas:
https://larepublica.pe/politica/2026/02/14/jaime-bayly-los-politicos-peruanos-sean-de-izquierda-o-de-derecha-siempre-encuentran-la-manera-de-decepcionarte-hnews-1110858
https://larepublica.pe/politica/2026/10/15/fuerza-popular-desconoce-votacion-que-pedia-debatir-los-limites-del-senado-sobre-el-pedido-de-facultades-hnews-1393335
https://larepublica.pe/politica/2026/10/15/ruth-luque-denuncia-que-el-fujimorismo-desconoce-votacion-en-comision-de-constitucion-no-aceptan-que-han-perdido-hnews-1036200
https://larepublica.pe/politica/2026/10/16/alejandro-aguinaga-viajo-a-argentina-el-martes-13-de-octubre-y-todavia-no-regresa-pese-a-debate-por-facultades-hnews-512448
https://larepublica.pe/politica/2026/10/16/alvaro-vargas-llosa-niega-intromision-politica-de-embajador-navarro-y-asegura-que-solo-tiene-una-voz-mas-participativa-hnews-763888


In [16]:
# PROCESO 3.6: Inspeccionar los campos de un artículo.
# Insumo: una de las URLs nuevas del proceso anterior.
# Resultado: campos disponibles y respaldo del HTML.

# webdriver: controla Chrome desde Python.
from selenium import webdriver

# By: permite localizar elementos de la página.
from selenium.webdriver.common.by import By

# WebDriverWait y EC: permiten esperar a que aparezca el titular.
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# BeautifulSoup: permite examinar el HTML recibido.
from bs4 import BeautifulSoup

# Elegir una URL de manera reproducible.
url_prueba = sorted(urls_nuevas)[0]
print("Artículo de prueba:", url_prueba)

navegador = webdriver.Chrome()

try:
    navegador.get(url_prueba)

    # Esperar a que aparezca el encabezado principal del artículo.
    WebDriverWait(navegador, 30).until(
        EC.visibility_of_element_located((By.TAG_NAME, "h1"))
    )

    html_articulo = navegador.page_source

    # Conservar el HTML antes de extraer información.
    (carpeta_raw / "articulo_prueba_selenium.html").write_text(
        html_articulo, encoding="utf-8"
    )

    pagina_articulo = BeautifulSoup(html_articulo, "html.parser")

    print("\nTitular visible:")
    print(pagina_articulo.find("h1").get_text(" ", strip=True))

    print("\nMetadatos del artículo:")
    for campo in [
        "og:title",
        "og:description",
        "article:published_time",
        "article:section"
    ]:
        elemento = pagina_articulo.find(
            "meta", attrs={"property": campo}
        )
        valor = elemento.get("content", "") if elemento else "No encontrado"
        print(f"{campo}: {valor}")

finally:
    navegador.quit()

Artículo de prueba: https://larepublica.pe/politica/2026/02/14/jaime-bayly-los-politicos-peruanos-sean-de-izquierda-o-de-derecha-siempre-encuentran-la-manera-de-decepcionarte-hnews-1110858

Titular visible:
Jaime Bayly: “Los políticos peruanos, sean de izquierda o de derecha, siempre encuentran la manera de decepcionarte”

Metadatos del artículo:
og:title: Jaime Bayly: “Los políticos peruanos, sean de izquierda o de derecha, siempre encuentran la manera de decepcionarte”
og:description: En esta entrevista, Bayly reflexiona sobre su trayectoria literaria, su relación con Vargas Llosa y la situación política en Perú, al tiempo que destaca el poder como un narcótico adictivo. La obra se enmarca en un contexto cambiante en Venezuela y Cuba, presentando a Chávez y Fidel Castro como figuras seductoras y despiadadas del poder.
article:published_time: No encontrado
article:section: No encontrado


In [17]:
# PROCESO 3.7: Buscar fecha y sección en el HTML conservado.
# Insumo: archivo articulo_prueba_selenium.html.
# Resultado: campos encontrados en los datos estructurados.

# json: convierte texto JSON en diccionarios y listas de Python.
import json

# BeautifulSoup: permite localizar los bloques dentro del HTML.
from bs4 import BeautifulSoup

html_guardado = (
    carpeta_raw / "articulo_prueba_selenium.html"
).read_text(encoding="utf-8")

pagina_guardada = BeautifulSoup(html_guardado, "html.parser")

bloques = pagina_guardada.find_all(
    "script", attrs={"type": "application/ld+json"}
)

print("Bloques JSON-LD encontrados:", len(bloques))

campos_interes = [
    "@type", "headline", "datePublished",
    "dateModified", "articleSection", "description"
]


def mostrar_campos(objeto):
    """Busca los campos también dentro de diccionarios y listas anidados."""
    if isinstance(objeto, dict):
        encontrados = {
            campo: objeto[campo]
            for campo in campos_interes
            if campo in objeto
        }

        if encontrados:
            print("\nObjeto encontrado:")
            for campo, valor in encontrados.items():
                print(f"{campo}: {str(valor)[:600]}")

        # Revisar también los objetos contenidos en este diccionario.
        for valor in objeto.values():
            if isinstance(valor, (dict, list)):
                mostrar_campos(valor)

    elif isinstance(objeto, list):
        for elemento in objeto:
            mostrar_campos(elemento)


for numero, bloque in enumerate(bloques, start=1):
    print(f"\n--- Bloque {numero} ---")

    try:
        datos_estructurados = json.loads(bloque.get_text())
        mostrar_campos(datos_estructurados)
    except json.JSONDecodeError:
        print("Este bloque no pudo interpretarse como JSON.")

Bloques JSON-LD encontrados: 7

--- Bloque 1 ---

Objeto encontrado:
@type: NewsMediaOrganization
description: Noticias del Perú y del mundo hoy. Últimas noticias de política, sociedad, economía, deportes, espectáculos, mundo, cine, tendencias, tecnología, salud y más.

Objeto encontrado:
@type: ContactPoint

--- Bloque 2 ---

Objeto encontrado:
@type: Organization

Objeto encontrado:
@type: Person

Objeto encontrado:
@type: PostalAddress

Objeto encontrado:
@type: ContactPoint

--- Bloque 3 ---

Objeto encontrado:
@type: WebSite

Objeto encontrado:
@type: SearchAction

Objeto encontrado:
@type: EntryPoint

--- Bloque 4 ---

Objeto encontrado:
@type: ItemList

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto encontrado:
@type: ListItem

Objeto 

In [18]:
# PROCESO 3.8: Localizar la posible bajada visible.
# Insumo: pagina_guardada, el HTML del artículo ya leído.
# Resultado: primeros subtítulos y párrafos posteriores al titular.

titular_html = pagina_guardada.find("h1")

if titular_html is None:
    print("No se encontró el titular principal.")
else:
    # Buscar los primeros cinco subtítulos o párrafos después del titular.
    candidatos_bajada = titular_html.find_all_next(
        ["h2", "p"], limit=5
    )

    for numero, elemento in enumerate(candidatos_bajada, start=1):
        print(f"\nCandidato {numero}")
        print("Etiqueta HTML:", elemento.name)
        print("Clases:", elemento.get("class", []))
        print("Texto:", elemento.get_text(" ", strip=True)[:1000])


Candidato 1
Etiqueta HTML: h2
Clases: ['MainContent_mainContent__teaser__vHB7m']
Texto: En esta entrevista exclusiva , Jaime Bayly reflexiona sobre su trayectoria literaria, su relación con Vargas Llosa y la situación política en Perú, al tiempo que destaca el poder como un narcótico adictivo. "Los golpistas", su nueva novela, se enmarca en un contexto cambiante en Venezuela y Cuba, presentando a Chávez y Fidel Castro como figuras seductoras y despiadadas del ejercicio del poder.

Candidato 2
Etiqueta HTML: p
Clases: []
Texto: La nueva novela de Jaime Bayly , Los golpistas (Galaxia Gutenberg / Revuelta), tiene todos los elementos de la marca de su narrativa: crítica, humor e ironía. Bayly narra lo que pasó con el presidente venezolano Hugo Chávez del 11 al 13 de abril de 2002, cuando fue detenido, mediante golpe militar, a razón de sus actitudes dictatoriales. En ese escenario, la figura del cubano Fidel Castro resultó clave. Pero también el reconocido autor peruano relata el proceso 

In [19]:
# PROCESO 3.9: Convertir el HTML de un artículo en un registro.
# Insumos: HTML del artículo y su URL.
# Resultado: una tabla de prueba con siete columnas.

# json: interpreta los datos estructurados del artículo.
import json

# pandas: organiza los registros en una tabla.
import pandas as pd

# BeautifulSoup: localiza elementos dentro del HTML.
from bs4 import BeautifulSoup


def extraer_articulo(html, url):
    """Transforma el HTML de un artículo en un diccionario de siete campos."""
    pagina = BeautifulSoup(html, "html.parser")
    articulos_encontrados = []

    # Buscar objetos NewsArticle, incluso dentro de listas o diccionarios.
    def buscar_articulos(objeto):
        if isinstance(objeto, dict):
            tipos = objeto.get("@type", [])
            if isinstance(tipos, str):
                tipos = [tipos]

            # Reconocer los dos tipos observados en nuestros artículos.
            tipos_admitidos = {"NewsArticle", "ReportageNewsArticle"}

            if tipos_admitidos.intersection(tipos):
                articulos_encontrados.append(objeto)
                
            for valor in objeto.values():
                if isinstance(valor, (dict, list)):
                    buscar_articulos(valor)

        elif isinstance(objeto, list):
            for elemento in objeto:
                buscar_articulos(elemento)

    for bloque in pagina.find_all(
        "script", attrs={"type": "application/ld+json"}
    ):
        try:
            buscar_articulos(json.loads(bloque.get_text()))
        except json.JSONDecodeError:
            continue

    # Evitar elegir silenciosamente un objeto si hay varios candidatos.
    if len(articulos_encontrados) != 1:
        raise ValueError(
            f"Se esperaba un NewsArticle; "
            f"se encontraron {len(articulos_encontrados)}."
        )

    datos_articulo = articulos_encontrados[0]

    titular = pagina.find("h1")

    # Usar el patrón observado para no confundir la bajada con otro h2.
    bajada = pagina.select_one('h2[class*="__teaser"]')

    seccion = datos_articulo.get("articleSection")

    return {
        "titular": titular.get_text(" ", strip=True) if titular else None,
        "bajada": bajada.get_text(" ", strip=True) if bajada else None,
        "medio": "La Republica",
        "seccion_original": seccion,
        "url": url,
        "fecha": datos_articulo.get("datePublished"),
        "etiqueta": "politica" if seccion == "Política" else None
    }


# Aplicar la función al artículo que ya inspeccionamos.
registro_prueba = extraer_articulo(html_guardado, url_prueba)
noticia_prueba = pd.DataFrame([registro_prueba])

print("Dimensiones:", noticia_prueba.shape)
print("\nValores faltantes por columna:")
print(noticia_prueba.isna().sum())

# Mostrar verticalmente para facilitar la lectura de los siete campos.
display(noticia_prueba.T)

Dimensiones: (1, 7)

Valores faltantes por columna:
titular             0
bajada              0
medio               0
seccion_original    0
url                 0
fecha               0
etiqueta            0
dtype: int64


,0
titular,"Jaime Bayly: “Los políticos peruanos, sean de ..."
bajada,"En esta entrevista exclusiva , Jaime Bayly ref..."
medio,La Republica
seccion_original,Política
url,https://larepublica.pe/politica/2026/02/14/jai...
fecha,2026-10-17T01:03:03-05:00
etiqueta,politica


In [21]:
# PROCESO 3.10: Verificar la extracción en otras tres noticias.
# Insumos: urls_nuevas y función extraer_articulo.
# Resultado: tabla noticias_piloto y lista errores_piloto.

# time: introduce pausas entre visitas.
import time

# hashlib: genera un identificador para guardar cada HTML sin colisiones.
import hashlib

# pandas: organiza los registros en una tabla.
import pandas as pd

# Selenium: abre Chrome, localiza elementos y espera su aparición.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Seleccionar tres URLs distintas del artículo ya inspeccionado.
urls_piloto = sorted(set(urls_nuevas) - {url_prueba})[:3]

registros_piloto = []
errores_piloto = []

navegador = webdriver.Chrome()

try:
    for numero, url in enumerate(urls_piloto, start=1):
        print(f"\nArtículo {numero} de {len(urls_piloto)}")
        time.sleep(2)

        try:
            navegador.get(url)

            WebDriverWait(navegador, 30).until(
                EC.visibility_of_element_located((By.TAG_NAME, "h1"))
            )

            html = navegador.page_source

            # Guardar el original antes de procesarlo.
            identificador = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
            archivo = carpeta_raw / f"piloto_{identificador}.html"
            archivo.write_text(html, encoding="utf-8")

            registro = extraer_articulo(html, url)
            registros_piloto.append(registro)

            print("Titular:", registro["titular"])
            print("Sección:", registro["seccion_original"])

        except (TimeoutException, ValueError) as error:
            errores_piloto.append({
                "url": url,
                "error": str(error)[:300]
            })
            print("No se pudo completar la extracción:", type(error).__name__)

finally:
    navegador.quit()

# Mantener las siete columnas incluso si ningún artículo pudo extraerse.
noticias_piloto = pd.DataFrame(
    registros_piloto,
    columns=list(registro_prueba.keys())
)

print("\nDimensiones:", noticias_piloto.shape)
print("Errores de extracción:", len(errores_piloto))

print("\nValores faltantes:")
print(noticias_piloto.isna().sum())

print("\nSecciones encontradas:")
print(noticias_piloto["seccion_original"].value_counts(dropna=False))

display(noticias_piloto)

if errores_piloto:
    display(pd.DataFrame(errores_piloto))


Artículo 1 de 3
Titular: Fuerza Popular desconoce votación que pedía debatir los límites del Senado sobre el pedido de facultades
Sección: Política

Artículo 2 de 3
Titular: Ruth Luque denuncia que el fujimorismo desconoce votación en Comisión de Constitución: "No aceptan que han perdido"
Sección: Política

Artículo 3 de 3
Titular: Alejandro Aguinaga viajó a Argentina el martes 13 de octubre y todavía no regresa pese a debate por facultades
Sección: Política

Dimensiones: (3, 7)
Errores de extracción: 0

Valores faltantes:
titular             0
bajada              0
medio               0
seccion_original    0
url                 0
fecha               0
etiqueta            0
dtype: int64

Secciones encontradas:
seccion_original
Política    3
Name: count, dtype: int64


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Fuerza Popular desconoce votación que pedía de...,Cuatro bancadas de oposición abandonaron la se...,La Republica,Política,https://larepublica.pe/politica/2026/10/15/fue...,2026-10-17T07:35:52-05:00,politica
1,Ruth Luque denuncia que el fujimorismo descono...,Tras abandonar la sesión junto a legisladores ...,La Republica,Política,https://larepublica.pe/politica/2026/10/15/rut...,2026-10-16T13:40:18-05:00,politica
2,Alejandro Aguinaga viajó a Argentina el martes...,"Según su registro en Migraciones, su salida de...",La Republica,Política,https://larepublica.pe/politica/2026/10/16/ale...,2026-10-17T09:02:58-05:00,politica


In [22]:
# PROCESO 3.12: Diagnosticar los HTML del piloto.
# Insumo: los tres archivos HTML guardados.
# Resultado: titular, bloques JSON-LD y tipos de objetos encontrados.

# hashlib: reconstruye el nombre usado al guardar cada HTML.
import hashlib

# json: interpreta los bloques de datos estructurados.
import json

# BeautifulSoup: permite examinar el HTML guardado.
from bs4 import BeautifulSoup


def obtener_tipos(objeto):
    """Recoge los valores de @type dentro de diccionarios y listas."""
    tipos = []

    if isinstance(objeto, dict):
        tipo = objeto.get("@type")

        if isinstance(tipo, str):
            tipos.append(tipo)
        elif isinstance(tipo, list):
            tipos.extend(str(valor) for valor in tipo)

        for valor in objeto.values():
            if isinstance(valor, (dict, list)):
                tipos.extend(obtener_tipos(valor))

    elif isinstance(objeto, list):
        for elemento in objeto:
            tipos.extend(obtener_tipos(elemento))

    return tipos


for numero, url in enumerate(urls_piloto, start=1):
    identificador = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
    archivo = carpeta_raw / f"piloto_{identificador}.html"

    html = archivo.read_text(encoding="utf-8")
    pagina = BeautifulSoup(html, "html.parser")

    print(f"\n--- Artículo {numero} ---")
    print("URL:", url)
    print("Tamaño del HTML:", len(html), "caracteres")

    h1 = pagina.find("h1")
    print("Titular:", h1.get_text(" ", strip=True) if h1 else "No encontrado")

    bloques = pagina.find_all(
        "script", attrs={"type": "application/ld+json"}
    )
    print("Bloques JSON-LD:", len(bloques))

    tipos = []
    for indice, bloque in enumerate(bloques, start=1):
        try:
            tipos.extend(obtener_tipos(json.loads(bloque.get_text())))
        except json.JSONDecodeError as error:
            print(f"Bloque {indice}: JSON no interpretable — {error.msg}")

    print("Tipos encontrados:", sorted(set(tipos)))

    # Comprobar si la fecha está escrita en alguna parte del HTML.
    print("Contiene el texto datePublished:", "datePublished" in html)


--- Artículo 1 ---
URL: https://larepublica.pe/politica/2026/10/15/fuerza-popular-desconoce-votacion-que-pedia-debatir-los-limites-del-senado-sobre-el-pedido-de-facultades-hnews-1393335
Tamaño del HTML: 651043 caracteres
Titular: Fuerza Popular desconoce votación que pedía debatir los límites del Senado sobre el pedido de facultades
Bloques JSON-LD: 7
Tipos encontrados: ['BreadcrumbList', 'ContactPoint', 'EntryPoint', 'ImageObject', 'ItemList', 'ListItem', 'NewsMediaOrganization', 'Organization', 'Person', 'PostalAddress', 'ReportageNewsArticle', 'SearchAction', 'SpeakableSpecification', 'WebPage', 'WebSite']
Contiene el texto datePublished: True

--- Artículo 2 ---
URL: https://larepublica.pe/politica/2026/10/15/ruth-luque-denuncia-que-el-fujimorismo-desconoce-votacion-en-comision-de-constitucion-no-aceptan-que-han-perdido-hnews-1036200
Tamaño del HTML: 630471 caracteres
Titular: Ruth Luque denuncia que el fujimorismo desconoce votación en Comisión de Constitución: "No aceptan que ha

In [23]:
# PROCESO 3.13: Reprocesar los tres HTML con la regla corregida.
# Insumo: respaldos del piloto y función extraer_articulo actualizada.
# Resultado: tabla de noticias y errores pendientes.

# hashlib: reconstruye los nombres de los respaldos.
import hashlib

# pandas: convierte los registros en una tabla.
import pandas as pd

registros_piloto = []
errores_piloto = []

for url in urls_piloto:
    identificador = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
    archivo = carpeta_raw / f"piloto_{identificador}.html"
    html = archivo.read_text(encoding="utf-8")

    try:
        registro = extraer_articulo(html, url)
        registros_piloto.append(registro)

    except ValueError as error:
        errores_piloto.append({
            "url": url,
            "error": str(error)
        })

noticias_piloto = pd.DataFrame(
    registros_piloto,
    columns=list(registro_prueba.keys())
)

print("Dimensiones:", noticias_piloto.shape)
print("Errores pendientes:", len(errores_piloto))

print("\nValores faltantes:")
print(noticias_piloto.isna().sum())

print("\nSecciones encontradas:")
print(noticias_piloto["seccion_original"].value_counts(dropna=False))

display(noticias_piloto)

if errores_piloto:
    display(pd.DataFrame(errores_piloto))

Dimensiones: (3, 7)
Errores pendientes: 0

Valores faltantes:
titular             0
bajada              0
medio               0
seccion_original    0
url                 0
fecha               0
etiqueta            0
dtype: int64

Secciones encontradas:
seccion_original
Política    3
Name: count, dtype: int64


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Fuerza Popular desconoce votación que pedía de...,Cuatro bancadas de oposición abandonaron la se...,La Republica,Política,https://larepublica.pe/politica/2026/10/15/fue...,2026-10-17T07:35:52-05:00,politica
1,Ruth Luque denuncia que el fujimorismo descono...,Tras abandonar la sesión junto a legisladores ...,La Republica,Política,https://larepublica.pe/politica/2026/10/15/rut...,2026-10-16T13:40:18-05:00,politica
2,Alejandro Aguinaga viajó a Argentina el martes...,"Según su registro en Migraciones, su salida de...",La Republica,Política,https://larepublica.pe/politica/2026/10/16/ale...,2026-10-17T09:02:58-05:00,politica


In [26]:
# PREPARACIÓN: Crear un navegador con menos interrupciones.
# Insumo: configuración de Chrome y lista de fuentes publicitarias.
# Resultado: navegador configurado antes de visitar el diario.

# webdriver: permite crear y configurar una sesión de Chrome.
from selenium import webdriver


def crear_navegador():
    opciones = webdriver.ChromeOptions()

    # Desactivar las notificaciones web.
    opciones.add_argument("--disable-notifications")

    # 2 significa bloquear estos permisos.
    opciones.add_experimental_option(
        "prefs",
        {
            "profile.default_content_setting_values.notifications": 2,
            "profile.default_content_setting_values.popups": 2
        }
    )

    # Restaurar el bloqueo normal de ventanas emergentes de Chrome.
    opciones.add_experimental_option(
        "excludeSwitches",
        ["disable-popup-blocking"]
    )

    web = webdriver.Chrome(options=opciones)

    try:
        # CDP: interfaz de Chrome para controlar funciones del navegador.
        # Activar el control de solicitudes antes de abrir la página.
        web.execute_cdp_cmd("Network.enable", {})

        # Lista inicial de fuentes publicitarias que no se descargarán.
        # No bloqueamos los artículos ni la consulta "Ver más noticias".
        web.execute_cdp_cmd(
            "Network.setBlockedURLs",
            {
                "urls": [
                    "*://*.googlesyndication.com/*",
                    "*://*.doubleclick.net/*",
                    "*://*.googleadservices.com/*",
                    "*://*.googletagservices.com/*"
                ]
            }
        )

    except Exception:
        # Evitar dejar Chrome abierto si falla la configuración.
        web.quit()
        raise

    return web

In [28]:
# PROCESO 4.1: Reunir URLs candidatas de Política.
# Insumos: sección Política y función obtener_urls_politica.
# Resultado: conjunto de URLs únicas y respaldos de cada bloque.

# time: introduce pausas entre clics.
import time

# Selenium: controla Chrome, localiza el botón y espera contenido nuevo.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    ElementClickInterceptedException
)

meta_urls = 100
maximo_clics = 12

# Recuperar los enlaces guardados en la ejecución anterior.
archivo_urls = carpeta_raw / "politica_urls_candidatas.txt"

urls_politica_candidatas = (
    set(archivo_urls.read_text(encoding="utf-8").splitlines())
    if archivo_urls.exists()
    else set()
)

print("URLs recuperadas:", len(urls_politica_candidatas))

# Exigir 12 URLs adicionales para probar el recorrido automático.
meta_urls = len(urls_politica_candidatas) + 12 

selector_boton = (
    "//*[self::button or self::a or @role='button']"
    "[contains(translate(normalize-space(.), "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), "
    "'ver más noticias')]"
)


navegador = crear_navegador()

try:
    navegador.get("https://larepublica.pe/politica")

    WebDriverWait(navegador, 30).until(obtener_urls_politica)

    # Bloque 0: contenido inicial.
    # Bloques siguientes: contenido después de cada clic.
    for bloque in range(maximo_clics + 1):
        urls_actuales = obtener_urls_politica(navegador)
        nuevas = urls_actuales - urls_politica_candidatas

        # update() agrega enlaces al conjunto sin repetir los existentes.
        urls_politica_candidatas.update(urls_actuales)

        print(
            f"Bloque {bloque}: {len(nuevas)} nuevas; "
            f"{len(urls_politica_candidatas)} acumuladas."
        )

        # Guardar el HTML correspondiente a este bloque.
        archivo = carpeta_raw / f"politica_listado_bloque_{bloque:02d}.html"
        archivo.write_text(navegador.page_source, encoding="utf-8")

        # Guardar también los enlaces después de cada avance.
        (carpeta_raw / "politica_urls_candidatas.txt").write_text(
            "\n".join(sorted(urls_politica_candidatas)),
            encoding="utf-8"
        )

        if len(urls_politica_candidatas) >= meta_urls:
            print("Meta de URLs candidatas alcanzada.")
            break

        if bloque == maximo_clics:
            print("Se alcanzó el límite de clics de esta prueba.")
            break

        # Buscar el botón de nuevo: la página puede haberlo reconstruido.
        boton = WebDriverWait(navegador, 20).until(
            EC.element_to_be_clickable((By.XPATH, selector_boton))
        )

        navegador.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            boton
        )


        # Conservar los enlaces visibles antes de activar el botón.
        urls_antes_del_clic = obtener_urls_politica(navegador)

        time.sleep(2)

        # Activar directamente el botón dentro de la página.
        # Así, el anuncio no recibe el clic destinado a "Ver más noticias".
        navegador.execute_script("arguments[0].click();", boton)

        # Verificar el resultado: debe aparecer alguna URL adicional.
        WebDriverWait(navegador, 30).until(
            lambda web: (
                obtener_urls_politica(web) - urls_antes_del_clic
            )
        )


except TimeoutException:
    print("Se detuvo: no apareció el botón o no llegaron URLs nuevas.")

except ElementClickInterceptedException as error:
    print("Se detuvo: otro elemento interceptó el clic.")
    print(error.msg[:700])

finally:
    navegador.quit()

print("\nTotal de URLs candidatas:", len(urls_politica_candidatas))

URLs recuperadas: 106
Bloque 0: 0 nuevas; 106 acumuladas.
Bloque 1: 0 nuevas; 106 acumuladas.
Bloque 2: 0 nuevas; 106 acumuladas.
Bloque 3: 0 nuevas; 106 acumuladas.
Bloque 4: 0 nuevas; 106 acumuladas.
Bloque 5: 0 nuevas; 106 acumuladas.
Bloque 6: 0 nuevas; 106 acumuladas.
Bloque 7: 0 nuevas; 106 acumuladas.
Bloque 8: 1 nuevas; 107 acumuladas.
Bloque 9: 12 nuevas; 119 acumuladas.
Meta de URLs candidatas alcanzada.

Total de URLs candidatas: 119


In [29]:
# PROCESO 4.2: Extraer los artículos de Política.
# Insumo: archivo de URLs candidatas y funciones ya verificadas.
# Resultado: tabla provisional, HTML de respaldo y errores registrados.

# time: introduce pausas entre visitas.
import time

# hashlib: genera nombres de archivo a partir de cada URL.
import hashlib

# pandas: organiza y guarda los registros como tabla.
import pandas as pd

# Selenium: espera el titular y reconoce errores de espera.
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Verificar las dependencias antes de abrir Chrome.
if not callable(globals().get("extraer_articulo")):
    raise RuntimeError("Ejecuta primero el Proceso 3.9 corregido.")

if not callable(globals().get("crear_navegador")):
    raise RuntimeError("Ejecuta primero la celda que define crear_navegador.")

# Leer el respaldo: no depender del conjunto que quedó en memoria.
archivo_urls = carpeta_raw / "politica_urls_candidatas.txt"
urls_recoleccion = sorted({
    url.strip()
    for url in archivo_urls.read_text(encoding="utf-8").splitlines()
    if url.strip()
})

columnas = [
    "titular", "bajada", "medio", "seccion_original",
    "url", "fecha", "etiqueta"
]

# Separar los resultados procesados de los respaldos originales.
carpeta_intermedia = carpeta_raw.parent / "intermedios"
carpeta_intermedia.mkdir(parents=True, exist_ok=True)

registros_politica = []
errores_politica = []

navegador = crear_navegador()

try:
    for numero, url in enumerate(urls_recoleccion, start=1):
        identificador = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
        archivo_html = carpeta_raw / f"articulo_{identificador}.html"
        archivo_piloto = carpeta_raw / f"piloto_{identificador}.html"

        try:
            # Reutilizar respaldos para evitar visitas innecesarias.
            if archivo_html.exists():
                html = archivo_html.read_text(encoding="utf-8")

            elif archivo_piloto.exists():
                html = archivo_piloto.read_text(encoding="utf-8")

            else:
                time.sleep(2)
                navegador.get(url)

                WebDriverWait(navegador, 30).until(
                    EC.visibility_of_element_located((By.TAG_NAME, "h1"))
                )

                html = navegador.page_source
                archivo_html.write_text(html, encoding="utf-8")

            registro = extraer_articulo(html, url)
            registros_politica.append(registro)

            print(f"{numero}/{len(urls_recoleccion)}: extraído")

        except (TimeoutException, ValueError) as error:
            errores_politica.append({
                "url": url,
                "error": str(error)[:500]
            })
            print(f"{numero}/{len(urls_recoleccion)}: pendiente de revisión")

        # Guardar el avance tras cada artículo.
        pd.DataFrame(
            registros_politica, columns=columnas
        ).to_csv(
            carpeta_intermedia / "politica_provisional.csv",
            index=False,
            encoding="utf-8-sig"
        )

        pd.DataFrame(
            errores_politica, columns=["url", "error"]
        ).to_csv(
            carpeta_intermedia / "politica_errores.csv",
            index=False,
            encoding="utf-8-sig"
        )

finally:
    navegador.quit()

noticias_politica_ampliadas = pd.DataFrame(
    registros_politica, columns=columnas
)

print("\nDimensiones:", noticias_politica_ampliadas.shape)
print("Errores:", len(errores_politica))

print("\nValores faltantes:")
print(noticias_politica_ampliadas.isna().sum())

print("\nSecciones:")
print(
    noticias_politica_ampliadas["seccion_original"]
    .value_counts(dropna=False)
)

print(
    "\nURLs duplicadas:",
    noticias_politica_ampliadas["url"].duplicated().sum()
)

display(noticias_politica_ampliadas.head())

1/119: extraído
2/119: extraído
3/119: extraído
4/119: extraído
5/119: extraído
6/119: extraído
7/119: extraído
8/119: extraído
9/119: extraído
10/119: extraído
11/119: extraído
12/119: extraído
13/119: extraído
14/119: extraído
15/119: extraído
16/119: extraído
17/119: extraído
18/119: extraído
19/119: extraído
20/119: extraído
21/119: extraído
22/119: extraído
23/119: extraído
24/119: extraído
25/119: extraído
26/119: extraído
27/119: extraído
28/119: extraído
29/119: extraído
30/119: extraído
31/119: extraído
32/119: extraído
33/119: extraído
34/119: extraído
35/119: extraído
36/119: extraído
37/119: extraído
38/119: extraído
39/119: extraído
40/119: extraído
41/119: extraído
42/119: extraído
43/119: extraído
44/119: extraído
45/119: extraído
46/119: extraído
47/119: extraído
48/119: extraído
49/119: extraído
50/119: extraído
51/119: extraído
52/119: extraído
53/119: extraído
54/119: extraído
55/119: extraído
56/119: extraído
57/119: extraído
58/119: extraído
59/119: extraído
60/119

,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,"Jaime Bayly: “Los políticos peruanos, sean de ...","En esta entrevista exclusiva , Jaime Bayly ref...",La Republica,Política,https://larepublica.pe/politica/2026/02/14/jai...,2026-10-17T01:03:03-05:00,politica
1,Carlos Infante: “La matanza de Uchuraccay es u...,"El periodista Carlos Infante , entre la memori...",La Republica,Política,https://larepublica.pe/politica/2026/08/15/car...,2026-10-18T10:33:36-05:00,politica
2,Raúl Tola: “Con los antecedentes del fujimoris...,"La última novela de Raúl Tola, El peso del air...",La Republica,Política,https://larepublica.pe/politica/2026/08/21/rau...,2026-10-15T05:00:00-05:00,politica
3,Confesiones del sicario que mató a “China Polo...,"Durante el interrogatorio policial, el venezol...",La Republica,Política,https://larepublica.pe/politica/2026/10/08/con...,2026-10-09T18:32:28-05:00,politica
4,Fugó cabecilla de la banda que mató a “China P...,En el mismo accidente de tránsito en Venezuela...,La Republica,Política,https://larepublica.pe/politica/2026/10/08/fug...,2026-10-09T18:41:01-05:00,politica


In [30]:
# PROCESO 4.3: Verificar el contenido de la tabla de Política.
# Insumo: noticias_politica_ampliadas.
# Resultado: diagnóstico de textos, fechas y titulares.

# pandas: permite revisar columnas y convertir textos en fechas.
import pandas as pd

print("Textos vacíos o compuestos solo por espacios:")
for columna in noticias_politica_ampliadas.columns:
    vacios = (
        noticias_politica_ampliadas[columna]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{columna}: {vacios}")

# Crear una serie de fechas sin cambiar la columna original.
# errors="coerce": convierte las fechas no interpretables en NaT.
# utc=True: expresa todas las fechas en una misma referencia horaria.
fechas_revision = pd.to_datetime(
    noticias_politica_ampliadas["fecha"],
    errors="coerce",
    utc=True
).dt.tz_convert("America/Lima")

print("\nFechas no interpretables:", fechas_revision.isna().sum())
print("Fecha mínima:", fechas_revision.min())
print("Fecha máxima:", fechas_revision.max())

# Comparar titulares ignorando mayúsculas y espacios sobrantes.
titulares_revision = (
    noticias_politica_ampliadas["titular"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.casefold()
)

print(
    "\nTitulares repetidos después de la primera aparición:",
    titulares_revision.duplicated().sum()
)

Textos vacíos o compuestos solo por espacios:
titular: 0
bajada: 0
medio: 0
seccion_original: 0
url: 0
fecha: 0
etiqueta: 0

Fechas no interpretables: 0
Fecha mínima: 2026-10-09 11:26:48-05:00
Fecha máxima: 2026-10-18 18:32:09-05:00

Titulares repetidos después de la primera aparición: 0


In [31]:
# PROCESO 5: Preparar la recolección para las otras secciones.
# Insumos: funciones verificadas y carpeta de respaldos.
# Resultado: funciones reutilizables para descubrir URLs y leer artículos.

# time: introduce pausas entre solicitudes.
import time

# re: reconoce las rutas de artículos de cada sección.
import re

# hashlib: genera nombres únicos para los respaldos HTML.
import hashlib

# urllib.parse: construye y separa direcciones web.
from urllib.parse import urljoin, urlparse

# BeautifulSoup: examina el HTML.
from bs4 import BeautifulSoup

# pandas: organiza, verifica y guarda las tablas.
import pandas as pd

# Selenium: localiza elementos y espera contenido.
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

# Comprobar dependencias antes de comenzar.
for nombre in ["crear_navegador", "extraer_articulo"]:
    if not callable(globals().get(nombre)):
        raise RuntimeError(f"Falta ejecutar la celda que define {nombre}.")

if "carpeta_raw" not in globals():
    raise RuntimeError("Falta definir carpeta_raw.")

columnas_noticias = [
    "titular", "bajada", "medio", "seccion_original",
    "url", "fecha", "etiqueta"
]

etiquetas_por_seccion = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia": "ciencia"
}

carpeta_intermedia = carpeta_raw.parent / "intermedios"
carpeta_intermedia.mkdir(parents=True, exist_ok=True)

selector_mas = (
    "//*[self::button or self::a or @role='button']"
    "[contains(translate(normalize-space(.), "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), "
    "'ver más noticias')]"
)


def obtener_urls_seccion(web, seccion):
    """Obtiene URLs únicas de artículos de la sección indicada."""
    pagina = BeautifulSoup(web.page_source, "html.parser")
    patron = rf"^/{re.escape(seccion)}/\d{{4}}/\d{{2}}/\d{{2}}/"
    urls = set()

    for enlace in pagina.find_all("a", href=True):
        partes = urlparse(urljoin(web.current_url, enlace["href"]))

        if (
            partes.hostname == "larepublica.pe"
            and re.match(patron, partes.path)
        ):
            urls.add(partes._replace(query="", fragment="").geturl())

    return urls


def reunir_urls(web, seccion, meta=110, maximo_clics=12):
    """Recupera enlaces guardados y amplía el conjunto hasta la meta."""
    archivo_urls = carpeta_raw / f"{seccion}_urls_candidatas.txt"

    urls = (
        {
            linea.strip()
            for linea in archivo_urls.read_text(
                encoding="utf-8"
            ).splitlines()
            if linea.strip()
        }
        if archivo_urls.exists()
        else set()
    )

    print(f"\n{seccion.upper()}: {len(urls)} URLs recuperadas.")

    if len(urls) >= meta:
        return sorted(urls)

    time.sleep(2)
    web.get(f"https://larepublica.pe/{seccion}")

    WebDriverWait(web, 30).until(
        lambda navegador: obtener_urls_seccion(navegador, seccion)
    )

    for bloque in range(maximo_clics + 1):
        visibles = obtener_urls_seccion(web, seccion)
        nuevas = visibles - urls
        urls.update(visibles)

        archivo_urls.write_text(
            "\n".join(sorted(urls)), encoding="utf-8"
        )

        (carpeta_raw / f"{seccion}_listado_{bloque:02d}.html").write_text(
            web.page_source, encoding="utf-8"
        )

        print(
            f"Bloque {bloque}: {len(nuevas)} nuevas; "
            f"{len(urls)} acumuladas."
        )

        if len(urls) >= meta or bloque == maximo_clics:
            break

        try:
            boton = WebDriverWait(web, 20).until(
                EC.element_to_be_clickable((By.XPATH, selector_mas))
            )

            time.sleep(2)
            web.execute_script("arguments[0].click();", boton)

            # Comparar con la vista anterior, no con todo el respaldo.
            WebDriverWait(web, 30).until(
                lambda navegador: (
                    obtener_urls_seccion(navegador, seccion) - visibles
                )
            )

        except TimeoutException:
            print("No apareció el botón o no llegaron enlaces nuevos.")
            break

    return sorted(urls)


def leer_articulo(web, url):
    """Reutiliza el HTML guardado o visita el artículo para obtenerlo."""
    identificador = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
    archivo = carpeta_raw / f"articulo_{identificador}.html"
    archivo_piloto = carpeta_raw / f"piloto_{identificador}.html"

    if archivo.exists():
        html = archivo.read_text(encoding="utf-8")

    elif archivo_piloto.exists():
        html = archivo_piloto.read_text(encoding="utf-8")

    else:
        time.sleep(2)
        web.get(url)

        WebDriverWait(web, 30).until(
            EC.visibility_of_element_located((By.TAG_NAME, "h1"))
        )

        html = web.page_source
        archivo.write_text(html, encoding="utf-8")

    registro = extraer_articulo(html, url)

    # Ampliar la clasificación sin modificar la función anterior.
    # Usar la sección declarada por el artículo, no solo su URL.
    seccion_original = registro.get("seccion_original")

    registro["etiqueta"] = (
        etiquetas_por_seccion.get(seccion_original.strip().casefold())
        if isinstance(seccion_original, str)
        else None
    )

    return registro


print("Funciones preparadas.")

Funciones preparadas.


In [32]:
# PROCESOS 5.1–5.3: Recolectar, extraer y verificar tres secciones.
# Insumo: funciones preparadas.
# Resultado: CSV provisionales, errores y resumen por sección.

tablas_otras_secciones = {}
resumen_secciones = []

navegador = crear_navegador()

try:
    for seccion in ["economia", "deportes", "ciencia"]:
        registros = []
        errores = []

        try:
            urls = reunir_urls(navegador, seccion)
        except (TimeoutException, WebDriverException) as error:
            print(f"No se pudo completar el listado de {seccion}:")
            print(str(error)[:300])
            continue

        for numero, url in enumerate(urls, start=1):
            try:
                registros.append(leer_articulo(navegador, url))

            except (TimeoutException, ValueError, WebDriverException) as error:
                errores.append({
                    "url": url,
                    "error": str(error)[:500]
                })

            # Guardar el avance después de cada artículo.
            tabla = pd.DataFrame(registros, columns=columnas_noticias)

            tabla.to_csv(
                carpeta_intermedia / f"{seccion}_provisional.csv",
                index=False,
                encoding="utf-8-sig"
            )

            pd.DataFrame(errores, columns=["url", "error"]).to_csv(
                carpeta_intermedia / f"{seccion}_errores.csv",
                index=False,
                encoding="utf-8-sig"
            )

            # Mostrar avance cada diez artículos y al finalizar.
            if numero % 10 == 0 or numero == len(urls):
                print(
                    f"{seccion}: {numero}/{len(urls)} revisados; "
                    f"{len(registros)} extraídos; {len(errores)} errores."
                )

        tabla = pd.DataFrame(registros, columns=columnas_noticias)
        tablas_otras_secciones[seccion] = tabla

        # Contar tanto nulos como textos vacíos o con solo espacios.
        faltantes = tabla.isna().copy()

        for columna in tabla.columns:
            faltantes[columna] |= (
                tabla[columna]
                .astype("string")
                .str.strip()
                .eq("")
                .fillna(False)
            )

        fechas = pd.to_datetime(
            tabla["fecha"], errors="coerce", utc=True
        )

        titulares = (
            tabla["titular"]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .str.casefold()
        )

        resumen_secciones.append({
            "seccion": seccion,
            "urls_candidatas": len(urls),
            "extraidas": len(tabla),
            "errores": len(errores),
            "filas_con_faltantes": int(faltantes.any(axis=1).sum()),
            "fechas_no_interpretables": int(fechas.isna().sum()),
            "urls_duplicadas": int(tabla["url"].duplicated().sum()),
            "titulares_repetidos": int(titulares.duplicated().sum()),
            "etiqueta_distinta_o_ausente": int(
                tabla["etiqueta"].fillna("").ne(seccion).sum()
            )
        })

        print("\nSecciones declaradas por los artículos:")
        print(tabla["seccion_original"].value_counts(dropna=False))

finally:
    navegador.quit()

resumen_recoleccion = pd.DataFrame(resumen_secciones)
display(resumen_recoleccion)


ECONOMIA: 0 URLs recuperadas.
Bloque 0: 10 nuevas; 10 acumuladas.
Bloque 1: 8 nuevas; 18 acumuladas.
Bloque 2: 9 nuevas; 27 acumuladas.
Bloque 3: 8 nuevas; 35 acumuladas.
Bloque 4: 9 nuevas; 44 acumuladas.
Bloque 5: 8 nuevas; 52 acumuladas.
Bloque 6: 10 nuevas; 62 acumuladas.
Bloque 7: 7 nuevas; 69 acumuladas.
Bloque 8: 10 nuevas; 79 acumuladas.
Bloque 9: 11 nuevas; 90 acumuladas.
Bloque 10: 7 nuevas; 97 acumuladas.
Bloque 11: 9 nuevas; 106 acumuladas.
Bloque 12: 11 nuevas; 117 acumuladas.
economia: 10/117 revisados; 10 extraídos; 0 errores.
economia: 20/117 revisados; 20 extraídos; 0 errores.
economia: 30/117 revisados; 30 extraídos; 0 errores.
economia: 40/117 revisados; 40 extraídos; 0 errores.
economia: 50/117 revisados; 50 extraídos; 0 errores.
economia: 60/117 revisados; 60 extraídos; 0 errores.
economia: 70/117 revisados; 70 extraídos; 0 errores.
economia: 80/117 revisados; 80 extraídos; 0 errores.
economia: 90/117 revisados; 90 extraídos; 0 errores.
economia: 100/117 revisados

,seccion,urls_candidatas,extraidas,errores,filas_con_faltantes,fechas_no_interpretables,urls_duplicadas,titulares_repetidos,etiqueta_distinta_o_ausente
0,economia,117,117,0,0,0,0,0,0
1,deportes,119,119,0,0,0,0,1,0
2,ciencia,113,112,1,0,0,0,0,0


In [33]:
# PROCESO 6.1: Integrar los resultados e inspeccionar incidencias.
# Insumo: cuatro CSV provisionales y registro de errores de Ciencia.
# Resultado: tabla conjunta y registros que requieren revisión.

# pandas: permite leer, unir y comparar las tablas.
import pandas as pd

secciones = ["politica", "economia", "deportes", "ciencia"]

tablas = [
    pd.read_csv(
        carpeta_intermedia / f"{seccion}_provisional.csv",
        encoding="utf-8-sig"
    )
    for seccion in secciones
]

# concat(): apila las filas de las cuatro tablas.
noticias_reunidas = pd.concat(tablas, ignore_index=True)

print("Dimensiones conjuntas:", noticias_reunidas.shape)
print("\nNoticias por categoría:")
print(noticias_reunidas["etiqueta"].value_counts())

# Identificar todas las filas implicadas en URLs repetidas.
urls_repetidas = noticias_reunidas["url"].duplicated(keep=False)

print("\nFilas con URL repetida:", urls_repetidas.sum())
if urls_repetidas.any():
    display(noticias_reunidas.loc[urls_repetidas])

# Comparar titulares ignorando mayúsculas y espacios sobrantes.
titulares_comparables = (
    noticias_reunidas["titular"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.casefold()
)

titulares_repetidos = titulares_comparables.duplicated(keep=False)

print("\nFilas con titular repetido:", titulares_repetidos.sum())
if titulares_repetidos.any():
    display(
        noticias_reunidas.loc[
            titulares_repetidos,
            ["titular", "bajada", "fecha", "etiqueta", "url"]
        ]
    )

print("\nError registrado en Ciencia:")
errores_ciencia = pd.read_csv(
    carpeta_intermedia / "ciencia_errores.csv",
    encoding="utf-8-sig"
)

for _, registro in errores_ciencia.iterrows():
    print("URL:", registro["url"])
    print("Error:", registro["error"])

Dimensiones conjuntas: (467, 7)

Noticias por categoría:
etiqueta
politica    119
deportes    119
economia    117
ciencia     112
Name: count, dtype: int64

Filas con URL repetida: 0

Filas con titular repetido: 2


,titular,bajada,fecha,etiqueta,url
247,Perú dejó una pobre imagen en su tercer amisto...,El equipo comandado por Mano Menezes casi ni r...,2026-10-03T15:09:50-05:00,deportes,https://larepublica.pe/deportes/2026/10/03/par...
250,Perú dejó una pobre imagen en su tercer amisto...,El equipo comandado por Mano Menezes casi ni r...,2026-10-03T15:09:50-05:00,deportes,https://larepublica.pe/deportes/2026/10/03/per...



Error registrado en Ciencia:
URL: https://larepublica.pe/ciencia/2026/10/02/los-primeros-pobladores-de-america-llegaron-durante-el-periodo-mas-frio-de-la-ultima-era-glacial-segun-estudio-de-adn-152528
Error: Se esperaba un NewsArticle; se encontraron 0.


In [4]:
# PROCESO 6.2: Recuperar las tablas y comparar titulares repetidos.
# Insumo: cuatro CSV provisionales guardados.
# Resultado: tabla conjunta y comparación de los registros repetidos.

# pathlib: permite localizar los archivos.
from pathlib import Path

# pandas: permite leer, unir y comparar tablas.
import pandas as pd

# Ruta donde se guardaron los resultados de esta tarea.
carpeta_intermedia = Path(
    r"d:\Mis Documentos\GitHub\Python-Intermedio-QLAB"
) / "tarea_1" / "datos" / "intermedios"

secciones = ["politica", "economia", "deportes", "ciencia"]

# Leer los cuatro archivos y apilar sus filas.
noticias_reunidas = pd.concat(
    [
        pd.read_csv(
            carpeta_intermedia / f"{seccion}_provisional.csv",
            encoding="utf-8-sig"
        )
        for seccion in secciones
    ],
    ignore_index=True
)

print("Dimensiones recuperadas:", noticias_reunidas.shape)

# Identificar titulares repetidos ignorando mayúsculas y espacios.
titulares_comparables = (
    noticias_reunidas["titular"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.casefold()
)

titulares_repetidos = titulares_comparables.duplicated(keep=False)
repetidas = noticias_reunidas.loc[titulares_repetidos].copy()

for indice, fila in repetidas.iterrows():
    print(f"\n--- Registro {indice} ---")
    for campo in ["titular", "bajada", "fecha", "url"]:
        print(f"{campo}: {fila[campo]}")

comparables = repetidas[
    ["titular", "bajada", "fecha", "etiqueta"]
].copy()

for campo in comparables.columns:
    comparables[campo] = (
        comparables[campo]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print(
    "\nCombinaciones distintas de titular, bajada, fecha y etiqueta:",
    len(comparables.drop_duplicates())
)

Dimensiones recuperadas: (467, 7)

--- Registro 247 ---
titular: Perú dejó una pobre imagen en su tercer amistoso: cayó 2-0 ante Canadá por la fecha FIFA 2026
bajada: El equipo comandado por Mano Menezes casi ni remató al arco rival y sigue sin lograr triunfos en la fecha FIFA de setiembre y octubre. Su próximo rival será Colombia.
fecha: 2026-10-03T15:09:50-05:00
url: https://larepublica.pe/deportes/2026/10/03/partido-peru-vs-canada-en-vivo-hoy-transmision-del-amistoso-por-fecha-fifa-2026-via-america-tv-y-america-tvgo-gratis-por-internet-139821

--- Registro 250 ---
titular: Perú dejó una pobre imagen en su tercer amistoso: cayó 2-0 ante Canadá por la fecha FIFA 2026
bajada: El equipo comandado por Mano Menezes casi ni remató al arco rival y sigue sin lograr triunfos en la fecha FIFA de setiembre y octubre. Su próximo rival será Colombia.
fecha: 2026-10-03T15:09:50-05:00
url: https://larepublica.pe/deportes/2026/10/03/peru-vs-canada-en-vivo-partido-amistoso-hoy-via-america-tv-175760



In [5]:
# PROCESO 6.2: Comparar las noticias con titular repetido.
# Insumo: noticias_reunidas y máscara titulares_repetidos.
# Resultado: comparación de contenido, sin eliminar registros.

repetidas = noticias_reunidas.loc[titulares_repetidos].copy()

for indice, fila in repetidas.iterrows():
    print(f"\n--- Registro {indice} ---")
    for campo in ["titular", "bajada", "fecha", "url"]:
        print(f"{campo}: {fila[campo]}")

# Comparar textos ignorando diferencias de espacios.
comparables = repetidas[["titular", "bajada", "fecha", "etiqueta"]].copy()

for campo in comparables.columns:
    comparables[campo] = (
        comparables[campo]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print(
    "\nCombinaciones distintas de titular, bajada, fecha y etiqueta:",
    len(comparables.drop_duplicates())
)


--- Registro 247 ---
titular: Perú dejó una pobre imagen en su tercer amistoso: cayó 2-0 ante Canadá por la fecha FIFA 2026
bajada: El equipo comandado por Mano Menezes casi ni remató al arco rival y sigue sin lograr triunfos en la fecha FIFA de setiembre y octubre. Su próximo rival será Colombia.
fecha: 2026-10-03T15:09:50-05:00
url: https://larepublica.pe/deportes/2026/10/03/partido-peru-vs-canada-en-vivo-hoy-transmision-del-amistoso-por-fecha-fifa-2026-via-america-tv-y-america-tvgo-gratis-por-internet-139821

--- Registro 250 ---
titular: Perú dejó una pobre imagen en su tercer amistoso: cayó 2-0 ante Canadá por la fecha FIFA 2026
bajada: El equipo comandado por Mano Menezes casi ni remató al arco rival y sigue sin lograr triunfos en la fecha FIFA de setiembre y octubre. Su próximo rival será Colombia.
fecha: 2026-10-03T15:09:50-05:00
url: https://larepublica.pe/deportes/2026/10/03/peru-vs-canada-en-vivo-partido-amistoso-hoy-via-america-tv-175760

Combinaciones distintas de titular

In [6]:
# PROCESO 6.3: Seleccionar y exportar la base final.
# Insumo: cuatro CSV provisionales.
# Resultado: 400 noticias, reserva y registro del duplicado retirado.

# pathlib: permite localizar archivos y crear la carpeta de salida.
from pathlib import Path

# pandas: permite integrar, comparar, ordenar y exportar las tablas.
import pandas as pd

carpeta_datos = (
    Path(r"d:\Mis Documentos\GitHub\Python-Intermedio-QLAB")
    / "tarea_1" / "datos"
)

carpeta_intermedia = carpeta_datos / "intermedios"
carpeta_salida = carpeta_datos / "procesados"

categorias = ["politica", "economia", "deportes", "ciencia"]
columnas = [
    "titular", "bajada", "medio", "seccion_original",
    "url", "fecha", "etiqueta"
]

noticias = pd.concat(
    [
        pd.read_csv(
            carpeta_intermedia / f"{categoria}_provisional.csv",
            encoding="utf-8-sig"
        )
        for categoria in categorias
    ],
    ignore_index=True
)

# Verificar datos completos antes de seleccionar.
if list(noticias.columns) != columnas:
    raise ValueError("Las columnas no coinciden con las siete esperadas.")

textos = noticias.astype("string")
vacios = textos.apply(lambda columna: columna.str.strip().eq(""))

if noticias.isna().any().any() or vacios.any().any():
    raise ValueError("Hay valores nulos o textos vacíos; revisar antes de exportar.")

if noticias["url"].duplicated().any():
    raise ValueError("Hay URLs repetidas; revisar antes de exportar.")

if set(noticias["etiqueta"]) != set(categorias):
    raise ValueError("Las categorías no coinciden con las cuatro esperadas.")

# Ordenar por URL determina cuál conservar cuando el contenido coincide.
ordenadas = noticias.sort_values("url").copy()

# Construir claves de comparación sin alterar los textos originales.
campos_clave = ["titular", "bajada", "fecha", "etiqueta"]
claves = ordenadas[campos_clave].astype("string").apply(
    lambda columna: columna.str.replace(
        r"\s+", " ", regex=True
    ).str.strip()
)

duplicados = claves.duplicated(keep="first")

# Detenerse si los datos difieren del caso que acabamos de revisar.
if int(duplicados.sum()) != 1:
    raise ValueError("Se esperaba un duplicado de contenido; revisar el diagnóstico.")

duplicado_retirado = ordenadas.loc[duplicados, columnas].copy()
sin_duplicados = ordenadas.loc[~duplicados].copy()

# Convertir fechas para ordenarlas cronológicamente.
# La columna original fecha permanece sin cambios.
sin_duplicados["_fecha_orden"] = pd.to_datetime(
    sin_duplicados["fecha"], errors="raise", utc=True
)

# Orden descendente por fecha; URL resuelve los empates.
ordenadas_por_fecha = sin_duplicados.sort_values(
    ["etiqueta", "_fecha_orden", "url"],
    ascending=[True, False, True]
)

conteos = ordenadas_por_fecha["etiqueta"].value_counts()
if any(conteos.get(categoria, 0) < 100 for categoria in categorias):
    raise ValueError("Alguna categoría tiene menos de 100 noticias.")

# head(100) por grupo conserva las 100 más recientes de cada categoría.
seleccion = ordenadas_por_fecha.groupby(
    "etiqueta", sort=False
).head(100)

noticias_finales = seleccion[columnas].reset_index(drop=True)

# Conservar las noticias no seleccionadas en una tabla de reserva.
reserva = ordenadas_por_fecha.loc[
    ~ordenadas_por_fecha.index.isin(seleccion.index),
    columnas
].reset_index(drop=True)

# Verificar antes de escribir.
if noticias_finales.shape != (400, 7):
    raise ValueError("La base final no tiene las dimensiones esperadas.")

if not noticias_finales["etiqueta"].value_counts().eq(100).all():
    raise ValueError("La distribución final no es de 100 por categoría.")

carpeta_salida.mkdir(parents=True, exist_ok=True)

archivo_final = carpeta_salida / "noticias_400.csv"

noticias_finales.to_csv(
    archivo_final, index=False, encoding="utf-8-sig"
)
reserva.to_csv(
    carpeta_intermedia / "noticias_reserva.csv",
    index=False, encoding="utf-8-sig"
)
duplicado_retirado.to_csv(
    carpeta_intermedia / "duplicado_contenido_retirado.csv",
    index=False, encoding="utf-8-sig"
)

# Volver a leer el CSV para verificar su escritura.
verificacion = pd.read_csv(archivo_final, encoding="utf-8-sig")
pd.testing.assert_frame_equal(verificacion, noticias_finales)

print("Dimensiones finales:", verificacion.shape)
print("\nNoticias por categoría:")
print(verificacion["etiqueta"].value_counts())
print("\nDuplicados de contenido retirados:", len(duplicado_retirado))
print("Noticias en reserva:", len(reserva))
print("Archivo final:", archivo_final)
print("Lectura de verificación: correcta.")

Dimensiones finales: (400, 7)

Noticias por categoría:
etiqueta
ciencia     100
deportes    100
economia    100
politica    100
Name: count, dtype: int64

Duplicados de contenido retirados: 1
Noticias en reserva: 66
Archivo final: d:\Mis Documentos\GitHub\Python-Intermedio-QLAB\tarea_1\datos\procesados\noticias_400.csv
Lectura de verificación: correcta.


# Analisis exploratorio (EDA)

In [1]:
# PROCESO 7.1: Preparar los datos para los gráficos.
# Insumo: noticias_400.csv.
# Resultado: tabla de análisis con fechas y longitud de titulares.

# pathlib: permite localizar el CSV.
from pathlib import Path

# pandas: organiza tablas, interpreta fechas y cuenta caracteres.
import pandas as pd

# Plotly Express: permite crear gráficos interactivos.
import plotly.express as px

archivo = (
    Path(r"d:\Mis Documentos\GitHub\Python-Intermedio-QLAB")
    / "tarea_1" / "datos" / "procesados" / "noticias_400.csv"
)

df = pd.read_csv(archivo, encoding="utf-8-sig")
noticias_eda = df.copy()

# Convertir las fechas a la hora de Perú.
noticias_eda["fecha_publicacion"] = pd.to_datetime(
    noticias_eda["fecha"], errors="raise", utc=True
).dt.tz_convert("America/Lima")

# Obtener el mes en formato año-mes para agrupar cronológicamente.
noticias_eda["mes"] = noticias_eda["fecha_publicacion"].dt.strftime("%Y-%m")

# Contar caracteres del texto guardado: incluye espacios y puntuación.
noticias_eda["longitud_titular"] = noticias_eda["titular"].str.len()

# Estilo compartido: fondo blanco y un solo color principal.
px.defaults.template = "simple_white"
px.defaults.color_discrete_sequence = ["#286A8A"]

print("Tabla original:", df.shape)
print(
    "Periodo:",
    noticias_eda["fecha_publicacion"].min(),
    "a",
    noticias_eda["fecha_publicacion"].max()
)
display(noticias_eda[["titular", "longitud_titular"]].head())

Tabla original: (400, 7)
Periodo: 2026-09-17 14:05:00-05:00 a 2026-10-18 19:32:25-05:00


,titular,longitud_titular
0,Los lobos gigantes modificados genéticamente p...,147
1,"En 1986, la explosión del reactor de Chernóbil...",164
2,Investigadores sudafricanos pintan franjas roj...,130
3,En 1993 se liberaron 300 bisontes en una prade...,130
4,Desconcertación entre los científicos: la Luna...,89


In [9]:
# Instalar o actualizar nbformat en el entorno activo del notebook.
# Plotly lo necesita para mostrar gráficos dentro de Jupyter.
%pip install --upgrade "nbformat>=4.2.0"


   ------ --------------------------------- 1/6 [fastjsonschema]
   -------------------- ------------------- 3/6 [jsonschema-specifications]
   -------------------------- ------------- 4/6 [jsonschema]
   -------------------------- ------------- 4/6 [jsonschema]
   -------------------------- ------------- 4/6 [jsonschema]
   --------------------------------- ------ 5/6 [nbformat]
   --------------------------------- ------ 5/6 [nbformat]
   --------------------------------- ------ 5/6 [nbformat]
   ---------------------------------------- 6/6 [nbformat]

Note: you may need to restart the kernel to use updated packages.


In [2]:
# PROCESO 8: Validar los extremos de longitud de los titulares.
# Insumo: noticias_eda, preparada en el Proceso 7.1.
# Resultado: cinco titulares más cortos y cinco más largos.

# pandas: permite ordenar y mostrar los textos completos.
import pandas as pd

columnas_revision = ["titular", "longitud_titular", "etiqueta"]

# Ordenar por longitud; la URL resuelve empates de forma reproducible.
titulares_cortos = noticias_eda.sort_values(
    ["longitud_titular", "url"],
    ascending=[True, True]
).head(5)

titulares_largos = noticias_eda.sort_values(
    ["longitud_titular", "url"],
    ascending=[False, True]
).head(5)

# Mostrar los titulares completos sin recortarlos con puntos suspensivos.
with pd.option_context("display.max_colwidth", None):
    print("CINCO TITULARES MÁS CORTOS")
    display(titulares_cortos[columnas_revision])

    print("\nCINCO TITULARES MÁS LARGOS")
    display(titulares_largos[columnas_revision])

print(
    "\nLongitud mínima:",
    noticias_eda["longitud_titular"].min()
)
print(
    "Longitud máxima:",
    noticias_eda["longitud_titular"].max()
)

CINCO TITULARES MÁS CORTOS


,titular,longitud_titular,etiqueta
291,Más allá de las cifras de pobreza,33,economia
360,Minería ilegal elevó 8 veces la demanda de combustible,54,politica
359,Vaticano informa que Papa León XIV tiene cáncer pulmonar,56,politica
134,¿Cuándo inicia la Liga Peruana de Vóley Femenina 2026-27?,57,deportes
378,Gobierno plantea regulación de armas de fuego de uso civil,58,politica



CINCO TITULARES MÁS LARGOS


,titular,longitud_titular,etiqueta
1,"En 1986, la explosión del reactor de Chernóbil obligó a que los perros fueran abandonados por sus dueños; décadas después, estos formaron grupos genéticos distintos",164,ciencia
84,"La humanidad roza sus límites: la población mundial alcanzaría los 12.400 millones en 2070, mientras científicos sugieren reducirla a 4.000 millones para 2200",158,ciencia
61,Kenia diseña la primera motocicleta eléctrica de producción con más de 1 km de autonomía por minuto de carga: su batería resiste el agua y cuenta con GPS,153,ciencia
107,"Pablo Guede ningunea a Franco Velazco, administrador de Universitario, tras polémicas declaraciones en su contra: ""No sé quién es, ¿habla de fútbol?""",149,deportes
32,"Australia recuperó sus praderas marinas tras un ciclón y grandes inundaciones, pero los científicos descubrieron que los peces ya no eran los mismos",148,ciencia



Longitud mínima: 33
Longitud máxima: 164


In [3]:
# PROCESO 7.2: Representar la distribución por categoría.
# Insumo: columna etiqueta.
# Resultado: gráfico de barras con el tamaño de cada categoría.

# value_counts(): transforma las noticias individuales en conteos.
conteo_categorias = (
    noticias_eda["etiqueta"]
    .value_counts()
    .rename_axis("categoria")
    .reset_index(name="noticias")
)

nombres_categorias = {
    "politica": "Política",
    "economia": "Economía",
    "deportes": "Deportes",
    "ciencia": "Ciencia"
}

conteo_categorias["categoria"] = (
    conteo_categorias["categoria"].map(nombres_categorias)
)

grafico_categorias = px.bar(
    conteo_categorias,
    x="categoria",
    y="noticias",
    text="noticias",
    title="La muestra contiene 100 noticias por categoría",
    labels={
        "categoria": "Categoría",
        "noticias": "Número de noticias"
    }
)

grafico_categorias.update_traces(
    textposition="outside",
    cliponaxis=False
)

grafico_categorias.update_layout(
    title_x=0,
    showlegend=False,
    yaxis_range=[0, conteo_categorias["noticias"].max() * 1.2]
)

grafico_categorias.show(renderer="browser")

In [4]:
# PROCESO 7.3: Representar la distribución temporal.
# Insumo: mes de publicación.
# Resultado: gráfico de noticias por mes con título calculado.

conteo_meses = (
    noticias_eda.groupby("mes")
    .size()
    .reset_index(name="noticias")
    .sort_values("mes")
)

# Incluir meses intermedios sin noticias, si los hubiera.
meses_completos = pd.period_range(
    start=noticias_eda["mes"].min(),
    end=noticias_eda["mes"].max(),
    freq="M"
).astype(str)

conteo_meses = (
    conteo_meses.set_index("mes")
    .reindex(meses_completos, fill_value=0)
    .rename_axis("mes")
    .reset_index()
)

maximo_mensual = conteo_meses["noticias"].max()
meses_maximos = conteo_meses.loc[
    conteo_meses["noticias"].eq(maximo_mensual), "mes"
]

porcentaje_maximo = 100 * maximo_mensual / len(noticias_eda)

# Distinguir un mes máximo de un empate.
if len(meses_maximos) == 1:
    titulo_fechas = (
        f"{meses_maximos.iloc[0]} reúne el "
        f"{porcentaje_maximo:.1f}% de las noticias de la muestra"
    )
else:
    titulo_fechas = (
        f"{len(meses_maximos)} meses empatan en el máximo: "
        f"{maximo_mensual} noticias por mes"
    )

grafico_fechas = px.bar(
    conteo_meses,
    x="mes",
    y="noticias",
    text="noticias",
    title=titulo_fechas,
    labels={
        "mes": "Mes de publicación (año-mes)",
        "noticias": "Número de noticias"
    }
)

grafico_fechas.update_traces(
    textposition="outside",
    cliponaxis=False
)

grafico_fechas.update_layout(
    title_x=0,
    showlegend=False,
    xaxis_type="category",
    yaxis_range=[0, maximo_mensual * 1.2]
)

grafico_fechas.show(renderer="browser")

In [5]:
# PROCESO 7.4: Representar la longitud de los titulares.
# Insumo: número de caracteres de cada titular.
# Resultado: histograma y referencia de la mediana.

# median(): obtiene el valor central de las longitudes ordenadas.
mediana_longitud = noticias_eda["longitud_titular"].median()

titulo_longitudes = (
    f"La mitad de los titulares tiene "
    f"{mediana_longitud:g} caracteres o menos"
)

grafico_longitudes = px.histogram(
    noticias_eda,
    x="longitud_titular",
    title=titulo_longitudes,
    labels={
        "longitud_titular": "Longitud del titular (caracteres)"
    }
)

# Fijar intervalos de 20 caracteres para facilitar la lectura.
limite_superior = (
    int(noticias_eda["longitud_titular"].max()) // 20 + 1
) * 20

grafico_longitudes.update_traces(
    xbins=dict(start=0, end=limite_superior, size=20),
    marker_line_color="white",
    marker_line_width=1
)

# Marcar la mediana sobre la distribución.
grafico_longitudes.add_vline(
    x=mediana_longitud,
    line_dash="dash",
    line_color="#A64B2A",
    annotation_text=f"Mediana: {mediana_longitud:g}",
    annotation_position="top right"
)

grafico_longitudes.update_layout(
    title_x=0,
    showlegend=False,
    bargap=0,
    yaxis_title="Número de noticias"
)

grafico_longitudes.show(renderer="browser")

In [6]:
# PROCESO 9: Exportar el CSV con el nombre exigido por la tarea.
# Insumo: datos/procesados/noticias_400.csv.
# Resultado: datos/noticias.csv, dimensiones y primeras cinco filas.

# pathlib: permite localizar los archivos.
from pathlib import Path

# pandas: permite leer, seleccionar columnas y exportar la tabla.
import pandas as pd

# display: muestra la tabla dentro del notebook.
from IPython.display import display

carpeta_datos = (
    Path(r"d:\Mis Documentos\GitHub\Python-Intermedio-QLAB")
    / "tarea_1" / "datos"
)

columnas_entrega = [
    "titular", "bajada", "medio", "seccion_original",
    "url", "fecha", "etiqueta"
]

# Leer la base final y conservar las siete columnas en el orden requerido.
df = pd.read_csv(
    carpeta_datos / "procesados" / "noticias_400.csv",
    encoding="utf-8-sig"
)[columnas_entrega]

if df.shape != (400, 7):
    raise ValueError(f"Dimensiones inesperadas: {df.shape}")

archivo_entrega = carpeta_datos / "noticias.csv"

# index=False evita guardar el índice como una octava columna.
# utf-8-sig facilita abrir tildes y eñes correctamente en Excel.
df.to_csv(
    archivo_entrega,
    index=False,
    encoding="utf-8-sig"
)

# Leer el archivo exportado y comprobar que conserva los datos.
verificacion = pd.read_csv(archivo_entrega, encoding="utf-8-sig")
pd.testing.assert_frame_equal(df, verificacion)

print("Archivo guardado:", archivo_entrega)
print("df.shape:", df.shape)
display(df.head())
print("Verificación del CSV: correcta.")

Archivo guardado: d:\Mis Documentos\GitHub\Python-Intermedio-QLAB\tarea_1\datos\noticias.csv
df.shape: (400, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Los lobos gigantes modificados genéticamente p...,Colossal Biosciences introdujo a estos animale...,La Republica,Ciencia,https://larepublica.pe/ciencia/2026/10/18/los-...,2026-10-18T19:32:25-05:00,ciencia
1,"En 1986, la explosión del reactor de Chernóbil...",Un estudio revela que estos animales cerca de ...,La Republica,Ciencia,https://larepublica.pe/ciencia/2026/10/18/en-1...,2026-10-18T18:37:52-05:00,ciencia
2,Investigadores sudafricanos pintan franjas roj...,Esta estrategia ayudaría a hacer más visibles ...,La Republica,Ciencia,https://larepublica.pe/ciencia/2026/10/18/inve...,2026-10-18T17:43:39-05:00,ciencia
3,En 1993 se liberaron 300 bisontes en una prade...,Estos grandes mamíferos ayudan a mantener la s...,La Republica,Ciencia,https://larepublica.pe/ciencia/2026/10/18/en-1...,2026-10-18T16:24:11-05:00,ciencia
4,Desconcertación entre los científicos: la Luna...,Nuestro satélite natural podría haberse formad...,La Republica,Ciencia,https://larepublica.pe/ciencia/2026/10/18/desc...,2026-10-18T15:19:19-05:00,ciencia


Verificación del CSV: correcta.


In [7]:
# Comprobar la dependencia necesaria para mostrar gráficos en el notebook.
import nbformat

print("Versión de nbformat:", nbformat.__version__)

Versión de nbformat: 5.11.1


In [10]:
# Mostrar el gráfico dentro de VS Code.
grafico_categorias.show(renderer="vscode")
grafico_fechas.show(renderer="vscode")
grafico_longitudes.show(renderer="vscode")